# NurtureJoy — DistilBERT Emotion Model (Lightweight Version)

This notebook trains the **6-class emotion model** for the NurtureJoy chatbot using the **smaller cleaner dataset** created by the subset notebook.

## Main changes vs heavier training
- trains on `nurturejoy_emotion_small.csv`
- max sequence length = **128**
- fp16 enabled when available
- small batch size + gradient accumulation
- weighted loss for class imbalance
- early stopping enabled
- saves the final model and tokenizer

## Expected input file
- `nurturejoy_emotion_small.csv`

## Expected columns
- `text`
- `category`

## Target labels
- `POSITIVE`
- `NEUTRAL`
- `ANXIETY`
- `STRESS`
- `LOW_MOOD`
- `HIGH_DISTRESS`


In [ ]:
# If needed in Colab, uncomment:
# !pip -q install transformers datasets evaluate accelerate scikit-learn pandas torch safetensors

import os
import re
import json
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)


In [ ]:
DATA_PATH = "nurturejoy_emotion_small.csv"
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "nurturejoy_distilbert_emotion_light"

assert os.path.exists(DATA_PATH), f"Missing {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
assert "text" in df.columns and "category" in df.columns, "Dataset must have text and category columns"

def clean_text(t: str) -> str:
    t = str(t).strip()
    t = re.sub(r"\s+", " ", t)
    return t

df = df.dropna(subset=["text", "category"]).copy()
df["text"] = df["text"].astype(str).map(clean_text)
df = df[df["text"].str.len() >= 5]

valid_labels = ["POSITIVE", "NEUTRAL", "ANXIETY", "STRESS", "LOW_MOOD", "HIGH_DISTRESS"]
df = df[df["category"].isin(valid_labels)].copy()

print("Shape:", df.shape)
print(df["category"].value_counts())
df.head()


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["category"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["category"]
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)


In [ ]:
label_list = ["POSITIVE", "NEUTRAL", "ANXIETY", "STRESS", "LOW_MOOD", "HIGH_DISTRESS"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

train_df["label"] = train_df["category"].map(label2id)
val_df["label"] = val_df["category"].map(label2id)
test_df["label"] = test_df["category"].map(label2id)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_ds, val_ds, test_ds


In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(range(len(label_list))),
    y=train_df["label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()


In [ ]:
eval_results = trainer.evaluate(test_ds)
print("Test metrics:", eval_results)

pred_output = trainer.predict(test_ds)
preds = np.argmax(pred_output.predictions, axis=-1)
y_true = pred_output.label_ids

print("\nClassification Report:")
print(classification_report(y_true, preds, target_names=label_list))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, preds))


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(os.path.join(OUTPUT_DIR, "label_mapping.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "label_list": label_list,
            "label2id": label2id,
            "id2label": id2label
        },
        f,
        indent=2
    )

print(f"Saved model + tokenizer to: {OUTPUT_DIR}")


## Next step
After this notebook finishes:

1. Download the whole output folder:
   - `nurturejoy_distilbert_emotion_light`

2. Put it into your backend here:
   - `dev/server/ml_models/distilbert_emotion/`

3. Restart the backend so the new `emotion_service.py` loads the transformer model instead of the old fallback model.
